# 01 PBSMT Training and Evaluation


## Purpose

This notebook generates Moses EMS configs from the original assignment template, then runs the PBSMT experiments.

The main experiment is a direct reuse of the original 5-gram template. The comparison experiments patch only a small set of active values: LM order, max phrase length, LM pruning, MERT n-best size, or evaluation search size.

The data preparation and SGM generation are not changed here. Run `00_data_and_sgm.ipynb` first if the SGM files need to be regenerated.

In [6]:
from pathlib import Path
import subprocess, re, shutil, os, difflib

BASE_DIR = Path('/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein')
G2P_DIR = BASE_DIR / 'g2p-par'
SGM_DIR = G2P_DIR / 'sgm'
MOSES_SRC_DIR = Path('/home/kyalkalay/tools/mosesdecoder')
MOSES_SCRIPT_DIR = MOSES_SRC_DIR / 'scripts'
MOSES_BIN_DIR = MOSES_SRC_DIR / 'bin'
EXPERIMENT_PERL = MOSES_SCRIPT_DIR / 'ems' / 'experiment.perl'
MTEVAL = MOSES_SCRIPT_DIR / 'generic' / 'mteval-v13a.pl'

# The original assignment experiment directory.
ORIGINAL_EXP_DIR = BASE_DIR / 'pbsmt-big-normalize'
ORIGINAL_TEMPLATE = ORIGINAL_EXP_DIR / 'config.baseline'
ORIGINAL_BASELINE_DIR = ORIGINAL_EXP_DIR / 'baseline'

# New clean experiment directory.
EXP_DIR = BASE_DIR / 'pbsmt_original_clean_experiments'

print('BASE_DIR:', BASE_DIR)
print('G2P_DIR:', G2P_DIR)
print('SGM_DIR:', SGM_DIR)
print('Original template:', ORIGINAL_TEMPLATE, ORIGINAL_TEMPLATE.exists())
print('Clean experiment dir:', EXP_DIR)

BASE_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein
G2P_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par
SGM_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par/sgm
Original template: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt-big-normalize/config.baseline True
Clean experiment dir: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments


## Helper functions

In [7]:
def sh(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, text=True, check=check)


def patch_template(
    template_text,
    order,
    max_phrase_length,
    lm_prune=None,
    nbest=None,
    eval_decoder_settings=None,
):
    """Patch only active experiment values in the original assignment template.

    Kept unchanged unless explicitly provided:
    - data paths
    - tokenizer/truecaser settings
    - training script
    - alignment settings
    - tuning decoder settings
    - evaluation SGM paths
    """
    text = template_text

    # Patch the active LM order line. Commented examples are ignored.
    text, n_order = re.subn(
        r'(?m)^order\s*=\s*\d+\s*$',
        f'order = {order}',
        text,
        count=1,
    )

    # Patch the active phrase extraction length line.
    text, n_phrase = re.subn(
        r'(?m)^max-phrase-length\s*=\s*\d+\s*$',
        f'max-phrase-length = {max_phrase_length}',
        text,
        count=1,
    )

    if n_order != 1 or n_phrase != 1:
        raise RuntimeError(f'Patch failed: order={n_order}, phrase={n_phrase}')

    # Optional: patch active LM pruning setting.
    if lm_prune is not None:
        text, n_prune = re.subn(
            r'(?m)^settings\s*=\s*"--prune\s+\'[^\']+\'(.*?)"\s*$',
            lambda m: f'settings = "--prune \'{lm_prune}\'{m.group(1)}"',
            text,
            count=1,
        )
        if n_prune != 1:
            raise RuntimeError(f'Patch failed: lm_prune={n_prune}')

    # Optional: patch active MERT n-best line.
    if nbest is not None:
        text, n_nbest = re.subn(
            r'(?m)^nbest\s*=\s*\d+\s*$',
            f'nbest = {nbest}',
            text,
            count=1,
        )
        if n_nbest != 1:
            raise RuntimeError(f'Patch failed: nbest={n_nbest}')

    # Optional: patch the active EVALUATION decoder-settings only.
    # We intentionally keep [TUNING] decoder-settings = "" unchanged.
    if eval_decoder_settings is not None:
        marker = '[EVALUATION]'
        idx = text.find(marker)
        if idx < 0:
            raise RuntimeError('Could not find [EVALUATION] section')
        before = text[:idx]
        after = text[idx:]
        after, n_eval_dec = re.subn(
            r'(?m)^decoder-settings\s*=\s*"[^"]*"\s*$',
            f'decoder-settings = "{eval_decoder_settings}"',
            after,
            count=1,
        )
        if n_eval_dec != 1:
            raise RuntimeError(f'Patch failed: eval_decoder_settings={n_eval_dec}')
        text = before + after

    return text


def write_configs(variants, directions=(('my', 'ph'), ('ph', 'my')), clean=False):
    assert ORIGINAL_TEMPLATE.exists(), f'Missing template: {ORIGINAL_TEMPLATE}'
    template = ORIGINAL_TEMPLATE.read_text(encoding='utf-8', errors='ignore')

    if clean and EXP_DIR.exists():
        shutil.rmtree(EXP_DIR)
    EXP_DIR.mkdir(parents=True, exist_ok=True)

    configs = []
    for v in variants:
        body = patch_template(
            template,
            order=v['order'],
            max_phrase_length=v['max_phrase_length'],
            lm_prune=v.get('lm_prune'),
            nbest=v.get('nbest'),
            eval_decoder_settings=v.get('eval_decoder_settings'),
        )
        for src, trg in directions:
            pair = f'{src}-{trg}'
            work = EXP_DIR / v['name'] / pair
            work.mkdir(parents=True, exist_ok=True)
            general = f"""# Config file for {pair} ({v['name']})

[GENERAL]
working-dir = {work}
input-extension = {src}
output-extension = {trg}
pair-extension = {pair}

"""
            cfg_path = work / f'config.baseline.{pair}'
            cfg_path.write_text(general + body, encoding='utf-8')
            configs.append(cfg_path)
    return configs


def run_configs(config_paths, no_graph=True):
    for cfg in config_paths:
        log = cfg.parent / 'run.log'
        cmd = f"perl {EXPERIMENT_PERL} -config {cfg} -exec {'-no-graph' if no_graph else ''} 2>&1 | tee {log}"
        sh(cmd, cwd=cfg.parent, check=False)

## Experiment set

The current baseline is `pbsmt_5gram_main`. The added tuning variants keep the original template and change only one small dimension at a time where possible.

Baseline target from the confirmed run:

```text
pbsmt_5gram_main / my-ph
BLEU-c = 75.9
Exact-match accuracy = 64.2398%
Exact-match errors = 1002 / 2802
```

In [8]:
EXPERIMENTS = [
    # Confirmed main baseline
    {
        'name': 'pbsmt_5gram_main',
        'order': 5,
        'max_phrase_length': 9,
        'purpose': 'confirmed main baseline',
    },

    # 5-gram phrase-length tuning
    {
        'name': 'pbsmt_5gram_phrase7',
        'order': 5,
        'max_phrase_length': 7,
        'purpose': 'shorter phrase extraction',
    },
    {
        'name': 'pbsmt_5gram_phrase11',
        'order': 5,
        'max_phrase_length': 11,
        'purpose': 'longer phrase extraction',
    },
    {
        'name': 'pbsmt_5gram_phrase13',
        'order': 5,
        'max_phrase_length': 13,
        'purpose': 'extra-long phrase extraction',
    },

    # 5-gram LM pruning and tuning variants
    {
        'name': 'pbsmt_5gram_less_pruned',
        'order': 5,
        'max_phrase_length': 9,
        'lm_prune': '0 0 0',
        'purpose': 'less LM pruning',
    },
    {
        'name': 'pbsmt_5gram_nbest200',
        'order': 5,
        'max_phrase_length': 9,
        'nbest': 200,
        'purpose': 'larger MERT n-best list',
    },
    {
        'name': 'pbsmt_5gram_search10000',
        'order': 5,
        'max_phrase_length': 9,
        'eval_decoder_settings': '-search-algorithm 1 -cube-pruning-pop-limit 10000 -s 10000',
        'purpose': 'stronger evaluation search',
    },

    # Context-size comparisons
    {
        'name': 'pbsmt_3gram_phrase5',
        'order': 3,
        'max_phrase_length': 5,
        'purpose': 'lower-context baseline',
    },
    {
        'name': 'pbsmt_3gram_phrase7',
        'order': 3,
        'max_phrase_length': 7,
        'purpose': 'lower-context phrase-length comparison',
    },
    {
        'name': 'pbsmt_7gram_phrase9',
        'order': 7,
        'max_phrase_length': 9,
        'purpose': 'higher-context comparison',
    },
    {
        'name': 'pbsmt_7gram_phrase11',
        'order': 7,
        'max_phrase_length': 11,
        'purpose': 'higher-context longer phrase comparison',
    },
]

configs = write_configs(EXPERIMENTS, clean=False)
for c in configs:
    print(c)

/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_main/my-ph/config.baseline.my-ph
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_main/ph-my/config.baseline.ph-my
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_phrase7/my-ph/config.baseline.my-ph
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_phrase7/ph-my/config.baseline.ph-my
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_phrase11/my-ph/config.baseline.my-ph
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_phrase11/ph-my/config.baseline.ph-my
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_phrase13/my-ph/config.baseline.my-ph
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsm

## Config sanity checks

In [9]:
main = EXP_DIR / 'pbsmt_5gram_main' / 'my-ph' / 'config.baseline.my-ph'
orig = ORIGINAL_BASELINE_DIR / 'my-ph' / 'config.baseline.my-ph'

# 1) Main config should retain the original important decoder settings.
text = main.read_text(encoding='utf-8')
assert 'decoder-settings = ""' in text, 'Tuning decoder settings changed unexpectedly'
assert 'decoder-settings = "-search-algorithm 1 -cube-pruning-pop-limit 5000 -s 5000"' in text, 'Evaluation decoder settings changed unexpectedly'
assert 'order = 5' in text
assert 'max-phrase-length = 9' in text
print('Main config decoder/order/phrase sanity passed.')

# 2) Search10000 should patch only evaluation search, not tuning decoder settings.
search_cfg = EXP_DIR / 'pbsmt_5gram_search10000' / 'my-ph' / 'config.baseline.my-ph'
search_text = search_cfg.read_text(encoding='utf-8')
assert 'decoder-settings = ""' in search_text
assert 'decoder-settings = "-search-algorithm 1 -cube-pruning-pop-limit 10000 -s 10000"' in search_text
print('Search10000 sanity passed.')

# 3) Optional diff against original generated config, if available.
def normalize(lines):
    out = []
    for x in lines:
        x = re.sub(r'working-dir\s*=.*', 'working-dir = <WORKDIR>', x)
        x = re.sub(r'^# Config file.*$', '# Config file <NORMALIZED>', x)
        out.append(x.rstrip())
    return out

if orig.exists():
    diff = list(difflib.unified_diff(
        normalize(orig.read_text(encoding='utf-8', errors='ignore').splitlines()),
        normalize(main.read_text(encoding='utf-8').splitlines()),
        fromfile='original', tofile='new', lineterm=''
    ))
    print('\n'.join(diff[:80]))
    print('diff lines:', len(diff))
else:
    print('Original generated config not found:', orig)

Main config decoder/order/phrase sanity passed.
Search10000 sanity passed.
--- original
+++ new
@@ -162,7 +162,6 @@
 
 # order of the language model
 order = 5
-
 ### tool to be used for training randomized language model from scratch
 # (more commonly, a SRILM is trained)
 #
@@ -411,7 +410,6 @@
 #
 #extract-settings = ""
 max-phrase-length = 9
-
 ### add extracted phrases from baseline model
 #
 #baseline-extract = $working-dir/model/extract.$baseline
diff lines: 18


## Run experiments

Start by keeping the confirmed baseline result, then run the tuning variants. For fastest iteration, run only `my-ph` first.

In [12]:
main_myph = [c for c in configs if 'pbsmt_5gram_main/my-ph' in str(c)]
all_myph = [c for c in configs if c.parent.name == 'my-ph']
tuning_myph = [c for c in all_myph if 'pbsmt_5gram_main/my-ph' not in str(c)]

# If you need to rerun the confirmed baseline:
run_configs(main_myph, no_graph=True)

# Recommended next step: run all my-ph tuning/comparison experiments except the already-confirmed baseline.
run_configs(tuning_myph, no_graph=True)

# Full run, both directions:
run_configs(configs, no_graph=True)

$ perl /home/kyalkalay/tools/mosesdecoder/scripts/ems/experiment.perl -config /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_main/my-ph/config.baseline.my-ph -exec -no-graph 2>&1 | tee /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_main/my-ph/run.log
perl: warning: Setting locale failed.
perl: warning: Please check that your locale settings:
	LANGUAGE = (unset),
	LC_ALL = (unset),
	LC_CTYPE = "C.UTF-8",
	LANG = "en_US.UTF-8"
    are supported and installed on your system.
perl: warning: Falling back to the standard locale ("C").
STARTING UP AS PROCESS 1314982 ON kyalkalay-HP AT Tue Jun  2 00:54:56 +0630 2026
LOAD CONFIG...
working directory is /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_main/my-ph
running experimental run number 3

ESTABLISH WHICH STEPS NEED TO BE RUN

FIND DEPENDENCIES BETWEEN STEPS

CHECKING IF OLD STEPS ARE RE

## Quick report preview

In [13]:
for report in sorted(EXP_DIR.glob('*/my-ph/evaluation/report.1')):
    print('\n###', report)
    print(report.read_text(encoding='utf-8', errors='ignore'))


### /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_3gram_phrase5/my-ph/evaluation/report.1
test: 75.89 (0.999) BLEU-c ; 75.89 (0.999) BLEU


### /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_3gram_phrase7/my-ph/evaluation/report.1
test: 75.93 (0.999) BLEU-c ; 75.93 (0.999) BLEU


### /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_less_pruned/my-ph/evaluation/report.1
test: 75.86 (1.000) BLEU-c ; 75.86 (1.000) BLEU


### /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_main/my-ph/evaluation/report.1
test: 75.90 (1.000) BLEU-c ; 75.90 (1.000) BLEU


### /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_nbest200/my-ph/evaluation/report.1
test: 75.69 (1.000) BLEU-c ; 75.69 (1.000) BLEU


### /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsm